In [1]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 23.43it/s]


Numba compilation complete!


In [5]:
#initial file processing
labcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = homecomp


filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\"
savedir = titledpath + filedir + "Compilation with delta\\2025deltagcollection\\"
saveosardir = titledpath + filedir + "Compilation with delta\\2025fallingtoosarcomp\\"
    
openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "ACR"
respondercsv = responder + ".csv"
wt = "w1118"


In [6]:
lstnew=[]

#lstnew should be the list of names you want to process the files with. Only choose one

#if you want to process all the names in the filedir

for file_no in os.listdir(openPath): 
    if respondercsv in file_no and "w1118" not in file_no :   
        f = os.path.join(openPath, file_no)
        dfe=pd.read_csv(f)
        exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
        driver = file_no.split(" ")[0]
        lstnew.append(driver)
#lst = lstnew.copy()
lst = [x for x in lstnew if x not in ['Th-Gal4', 'R58']]

#processing ONLY specific names
# lst = ["MB112C"]

print(lst)

['MB011B', 'MB018B', 'MB027B', 'MB057B', 'MB077B', 'MB080C', 'MB082C', 'MB083C', 'MB093C', 'MB112C', 'MB210B', 'MB242A', 'MB310C', 'MB319C', 'MB323B', 'MB399B', 'MB434B', 'MB542B', 'MB543B', 'R76B09', 'SS01127', 'SS01188', 'SS01298', 'SS01308', 'SS01337', 'SS01388', 'SS46348', 'SS52050', 'SS67662', 'SS67727', 'SS67741', 'SS75199', 'SS75200', 'SS76094', 'SS77383', 'SS77424', 'SS77442', 'SS77450', 'SS80896', 'SS80958', 'SS81353', 'SS81521', 'SS86947', 'SS95118', 'SS97567', 'VT999036']


In [ ]:
for n in lst:
    driver = n
    print(n)
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    #adjust this depending on timeframe
    dfexpt = NLCLIMB.timerule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.timerule(NLCLIMB.generation(wtdf, wt))
       
    #processing before dabest application 
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True)
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_pp = NLMATH.bheight(NLMATH.pauseheight(dfexpt), NLMATH.pauseheight(dfwt)).reset_index(drop=True)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    df_sim = pd.concat([NLMATH.straightnessindexmeter(dfexpt, "Expt"), NLMATH.straightnessindexmeter(dfwt, "WT")], axis = 0).reset_index(drop=True)

    #pause and bouts
    wttotalmeanevent, wttotalnumberevent = NLMATH.pausecomp(dfwt, wt)
    expttotalmeanevent, expttotalnumberevent = NLMATH.pausecomp(dfexpt, driver)
    alltgtmeandf_bout = pd.concat([NLMATH.pausenumber(wttotalmeanevent, n, "Bouts"), NLMATH.pausenumber(expttotalmeanevent, n, "Bouts")], axis = 0).reset_index(drop=True)
    alltgtnumberdf_bout = pd.concat([NLMATH.pausenumber(wttotalnumberevent, n, "Bouts"), NLMATH.pausenumber(expttotalnumberevent, n, "Bouts")], axis = 0).reset_index(drop=True)
            
    #___________________________________________#    
    # meandiff plots -- you run mean_diff instead of delta_g because since all the binary data is at the same dimension, no standardization is required and empirical delta delta is sufficient
    #dff2_prop = NLMATH.deltaversion_meandiff(df_f, "binary_fallvalue", "fallprop")
    dff2_number = NLMATH.deltaversion_meandiff(df_f, "Fall", "fallnumber")   #number of falls is under deltaversion_binary because the number of flies that fall could actually be so few in number, that the SD is 0, and thus hedges g will not be able to perform since the divisor ==0

    #deltag plots
    dfs2 = NLMATH.deltaversion_deltag(df_sp, "Velocity", "speed")
    dfh2 = NLMATH.deltaversion_deltag(df_h, "Y", "height")
    dfbs2 = NLMATH.deltaversion_deltag(df_bsp, "BSpeed", "bspeed")
    dfpp2 = NLMATH.deltaversion_deltag(df_pp, "Height", "pausepos")
    dfmv2 = NLMATH.deltaversion_deltag(df_maxv, "maxvelocity", "maxvelocity")
    dfsim2 = NLMATH.deltaversion_deltag(df_sim, "averagestraightnessindex", "straightindex")
    #pause and bouts
    dfmb2 = NLMATH.deltaversion_deltag(alltgtmeandf_bout, "Bouts", "meanbout")     
    dfnb2 = NLMATH.deltaversion_deltag(alltgtnumberdf_bout, "Bouts", "bout")
    
    #singledelta processing
    lsr_bsp = NLMATH.log2metric(df_bsp, "BSpeed")
    lsr_sp = NLMATH.log2metric(df_sp, 'Velocity')    
    
    #new index and ratio metrics
    bout_index_nb = NLMATH.boutindex(alltgtnumberdf_bout, "Bouts") 
    ratio_mb = NLMATH.simplemetricratio(alltgtmeandf_bout, "Bouts") 
    ratio_maxv = NLMATH.simplemetricratio(df_maxv, "maxvelocity")
    

    df_lsrbsp = NLMATH.singledelta(lsr_bsp, "log2 BSpeed", "log2bspeed")
    df_lsrsp = NLMATH.singledelta(lsr_sp, "log2 Velocity", "log2speed")
    df_boutindex_nb = NLMATH.singledelta(bout_index_nb, "Bouts", "boutnumber_index")  # Renamed from ratio to index
    df_ratio_mb = NLMATH.singledelta(ratio_mb, "Bouts", "boutduration_ratio")
    df_ratio_maxv = NLMATH.singledelta(ratio_maxv, "maxvelocity", "maxvelocity_ratio")
    
    #final df and saving into excel
    dftotal = pd.concat([dff2_number, dfs2, dfh2, dfbs2, dfpp2, dfmv2, dfsim2, dfmb2, dfnb2, df_lsrbsp, df_lsrsp, df_boutindex_nb, df_ratio_mb, df_ratio_maxv], axis = 1)
    dftotal['MBON'] = n
    dftotal.set_index("MBON", inplace = True)
    dftotal.to_csv(savedir + n + " x " + responder + "_deltag_allstats.csv")
    
print("Done!")        

MB011B
MB018B
MB027B
MB057B
MB077B
MB080C
MB082C
MB083C
MB093C
MB112C
MB210B
MB242A
MB310C
MB319C
MB323B
MB399B
MB434B
MB542B
MB543B
R76B09
SS01127
SS01188
SS01298
SS01308
SS01337
SS01388
SS46348
SS52050
SS67662
SS67727
SS67741
SS75199
SS75200
SS76094
SS77383
SS77424
SS77442
SS77450
SS80896
SS80958
SS81353
SS81521
SS86947
SS95118
SS97567
VT999036
Done!
